# Tweedie-modell for ren egen-skadepremie

Denne notebooken setter opp en samlet Tweedie-GLM for forventet årlig
egen-skadekostnad. Den bruker bare utviklingsårene 2022–2023. Prediktorer og
CV-struktur holdes låst mens Tweedie-kraften `p` sammenlignes.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import GroupKFold

from src_core_glm.glm_core import (
    cross_validate_glm,
    fit_glm,
    glm_spec,
    prepare_design_frame,
)
from src_core_glm.model_data import (
    TRAIN_YEARS,
    assert_development_years,
    build_development_frames,
    to_model_frame,
)
from src_core_glm.model_selection import run_backward_ablation, select_candidate_stage

SEED = 100
N_FOLDS = 5
FIT_SETTINGS = {"maxiter": 200, "tol": 1e-8}
POWER_GRID = (1.1, 1.3, 1.5, 1.7, 1.9)

## 1. Datagrunnlag

For poliseår $i$ modelleres samlet kostnad per eksponeringsår som

$$
Y_i/e_i \sim \operatorname{Tweedie}(\mu_i,\phi e_i^{1-p}),
\qquad \log(\mu_i)=\beta_0+\sum_j\beta_jx_{ij},
$$

med `total_exposure` som frekvens-/eksponeringsvekt. For $1<p<2$ har
fordelingen både masse i null og en kontinuerlig positiv del.

In [ ]:
frames = build_development_frames()
development = frames["development"]
assert_development_years(development)

PREDICTORS = [
    "policy_type",
    "year",
    "driver_age",
    "log_vehicle_value",
    "performance_hp_per_tonne",
    "fuel_type",
    "circulation_area",
    "municipality_type",
    "payment_frequency",
    "business_type",
    "vehicle_brand_pooled",
    "seats",
]
model_columns = [
    "insured_id",
    "total_exposure",
    "property_incurred",
    *PREDICTORS,
]
model_frame = to_model_frame(development, model_columns)
model_frame["pure_premium"] = (
    model_frame["property_incurred"] / model_frame["total_exposure"]
)

## 2. Gruppebasert CV

Samme `insured_id` ligger ikke i både trening og validering. Dette er samme
foldstruktur som i de øvrige GLM-fasene; 2024 inngår ikke i rammen.

In [ ]:
group_kfold = GroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cv_folds = [
    {
        "fold": f"gruppe_{number + 1}",
        "train_index": model_frame.index[train_positions],
        "val_index": model_frame.index[val_positions],
    }
    for number, (train_positions, val_positions) in enumerate(
        group_kfold.split(model_frame, groups=model_frame["insured_id"])
    )
]

## 3. Tweedie-spesifikasjon

`glm_spec` og `cross_validate_glm` er generiske: eneste modellparameter som
endres her er `power`. Formelen og vekten er identiske for alle kandidater.

In [ ]:
def build_tweedie_specification(power, predictors=PREDICTORS, name=None):
    """Bygg én samlet Tweedie-GLM for en forhåndsdefinert kraftparameter."""
    family = sm.families.Tweedie(
        var_power=power,
        link=sm.families.links.Log(),
    )
    return glm_spec(
        name=name or f"tweedie_p_{power:.1f}",
        x=predictors,
        y="pure_premium",
        data=model_frame,
        family=family,
        weight="total_exposure",
        power=power,
    )


tweedie_specs = {
    power: build_tweedie_specification(power) for power in POWER_GRID
}

## 4. CV-oppsett for `p`

Hver kandidat får samme fold, preprocessing og foldkontroller. Valg av `p`
skal senere baseres på pooled OOF-Tweedie-deviance, ikke på treningsscore.

In [ ]:
cv_results = {
    power: cross_validate_glm(
        specification,
        model_frame,
        cv_folds,
        fit_kwargs=FIT_SETTINGS,
    )
    for power, specification in tweedie_specs.items()
}


def summarize_power_cv(cv_results):
    """Samle foldscore til én sammenlignbar, eksponeringsvektet oversikt."""
    rows = []
    for power, result in cv_results.items():
        scores = result["scores"]
        rows.append(
            {
                "power": power,
                "pooled_oof_deviance": np.average(
                    scores["val_deviance"], weights=scores["val_weight"]
                ),
                "mean_fold_deviance": scores["val_deviance"].mean(),
                "valid": result["valid"],
            }
        )
    return pd.DataFrame(rows).sort_values("pooled_oof_deviance")


power_cv_summary = summarize_power_cv(cv_results)

## 5. Finalist

Denne cellen velger foreløpig laveste pooled OOF-deviance. Eventuell
vurdering av flat scorekurve og praktisk avrunding dokumenteres før sluttfit.

In [ ]:
selected_power = power_cv_summary.iloc[0]["power"]
selected_specification = tweedie_specs[selected_power]

## 6. Variabelseleksjon med valgt `p`

Etter at `p` er valgt, holdes den fast. Deretter brukes den eksisterende
treleddsregelen for kandidatvariabler: lavere pooled OOF-deviance, forbedring
i minst fire av fem folder og gyldig fit i alle folder. Dette skiller
hyperparameter-valget fra variabelseleksjonen.

In [ ]:
selected_power = float(selected_power)
selection_specifications = {
    "tweedie_full": build_tweedie_specification(
        selected_power, name="tweedie_full"
    )
}
selection_results = {
    "tweedie_full": cross_validate_glm(
        selection_specifications["tweedie_full"],
        model_frame,
        cv_folds,
        fit_kwargs=FIT_SETTINGS,
    )
}


def select_tweedie_stage(parent_name, candidate_names):
    """Bruk variabelseleksjonsregelen med valgt Tweedie-kraft."""
    return select_candidate_stage(
        selection_results,
        parent_name,
        candidate_names,
        model_frame["pure_premium"],
        model_frame["total_exposure"],
        power=selected_power,
    )


def build_tweedie_removal(parent_name, column):
    """Bygg en kandidat som fjerner én råprediktor fra foreldreformelen."""
    parent_predictors = selection_specifications[parent_name]["x"]
    predictors = [predictor for predictor in parent_predictors if predictor != column]
    return build_tweedie_specification(
        selected_power,
        predictors=predictors,
        name=f"{parent_name}_uten_{column}",
    )


selected_variable_model, removed_variables, variable_selection_tables = run_backward_ablation(
    start_name="tweedie_full",
    specifications=selection_specifications,
    cv_results=selection_results,
    protected_predictors=("policy_type", "year"),
    build_candidate=build_tweedie_removal,
    evaluate_candidate=lambda specification: cross_validate_glm(
        specification,
        model_frame,
        cv_folds,
        fit_kwargs=FIT_SETTINGS,
    ),
    select_stage=select_tweedie_stage,
)

# Sluttfit kjøres først når både `p` og variabelseleksjonen er gjennomgått.